In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [2]:
# Datasets & Dataloaders

from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# Convert images to scale(0,1) & normalise(-1,1)  (why -1,1 bcoz the value will range btw these lowest -1 and highest values 1)
transform = transforms.Compose([
    transforms.ToTensor(), # converts to tensor and scales (0, 1)
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # for RGB 3 times 0.5, which shifts image values from [0,1] → [-1,1]
])

# CIFAR10 already consists of train dataset fnx
train_dataset = CIFAR10(root="./data", train=True, download=True, transform=transform)
test_dataset = CIFAR10(root="./data", train=False, download=True, transform=transform)


In [3]:
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=64)

# Build CNN

In [4]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        # Convolution layers
        self.conv_layers = nn.Sequential(
            # convolution layer 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1), # i/p = 3, feature_map = 32
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # kernel size = 2, stride = 2
            # convolution layer 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            # convolution layer 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )

        # Fully connected layers
        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),  # check notes why this
            nn.ReLU(),

            nn.Linear(256, 10),
        )
    # forward prop...
    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)  # flattening
        x = self.fc_layers(x)
        return x

In [5]:
model = CNN()

# loss & optimizers
criterion = nn.CrossEntropyLoss()  # bcoz it is a multi-class classification
optimizer = optim.Adam(model.parameters())

In [6]:
# Train CNN

epochs = 10
for epoch in range(epochs):
    model.train()
    epoch_training_loss = 0.0

    for xb, yb in train_dataloader: # xb is images and yb is o/p
        optimizer.zero_grad()
        
        outputs = model.forward(xb)  # forward prop..
        loss = criterion(outputs, yb)  # loss computation
        loss.backward() # backward prop...
        optimizer.step() # update the params

        epoch_training_loss += loss.item()
    print(f"Epoch: {epoch+1} / {epochs} & Loss: {epoch_training_loss / len(train_dataloader)}")

Epoch: 1 / 10 & Loss: 1.3849279888145758
Epoch: 2 / 10 & Loss: 0.9546248693295452
Epoch: 3 / 10 & Loss: 0.7677657893856468
Epoch: 4 / 10 & Loss: 0.6450050104304653
Epoch: 5 / 10 & Loss: 0.5356819851090536
Epoch: 6 / 10 & Loss: 0.4526231628664009
Epoch: 7 / 10 & Loss: 0.3635219913881148
Epoch: 8 / 10 & Loss: 0.2914090118825893
Epoch: 9 / 10 & Loss: 0.2285559702988552
Epoch: 10 / 10 & Loss: 0.180570517025907


In [9]:
# Evaluate our CNN

total = 0
correct = 0

model.eval()

with torch.no_grad():
    for x, y in test_dataloader:
        outputs = model.forward(x)
        _, predicted = torch.max(outputs, 1)

        correct += (predicted == y).sum().item()
        total += y.size(0)
print("Accuracy:", correct / total)

Accuracy: 0.7437
